# TSRNet — ECG Anomaly Detection Scores on PTB-XL (Auto-Merge Version)

This notebook takes your **raw PTB-XL dataset** (split into multiple zip files by Google Drive) and runs it through **TSRNet** to produce anomaly-detection scores (ROC AUC) on the test set.

**What this notebook does differently:**
It includes an automatic Python script to hunt down your split Kaggle datasets, merge them together into a single folder, and automatically configure the path. You don't have to change anything manually!

**Before running:** go to `Runtime -> Change runtime type -> GPU` (T4 x2 is highly recommended).

In [ ]:
import urllib.request

try:
    urllib.request.urlopen('https://pypi.org', timeout=3)
    print('✅ Internet is ON!')
except:
    print('\n🚨 CRITICAL ERROR: KAGGLE INTERNET IS TURNED OFF! 🚨')
    print('You cannot install dependencies without internet.')
    print('\n👉 HOW TO FIX THIS:')
    print('1. Look at the right sidebar in Kaggle (Session options).')
    print('2. Toggle "Internet" to ON.')
    print('3. After turning it on, run this cell again!\n')
    raise Exception("KAGGLE INTERNET IS OFF. Follow the instructions printed above to turn it on!")

In [ ]:
import os
import shutil

target_dir = "/kaggle/working/PTB-XL"
os.makedirs(target_dir, exist_ok=True)
print("Hunting for your files and merging them together... (this might take a minute)")

for root, dirs, files in os.walk("/kaggle/input"):
    # Copy the CSV files
    for file in files:
        if file in ["ptbxl_database.csv", "scp_statements.csv"]:
            shutil.copy(os.path.join(root, file), target_dir)
            
    # Merge the records500 folders using Python's ultra-fast built-in merger
    if "records500" in dirs:
        src = os.path.join(root, "records500")
        dst = os.path.join(target_dir, "records500")
        print(f"Merging chunks into {dst}...")
        # dirs_exist_ok=True allows us to merge multiple split folders seamlessly and instantly!
        shutil.copytree(src, dst, dirs_exist_ok=True)

PTBXL_DATA_PATH = target_dir
print("✅ Merge complete! Your dataset is fully assembled.")
print("Target Path Configured:", PTBXL_DATA_PATH)

In [ ]:
# 2. Install dependencies
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb", "heartpy", "PyWavelets", "tqdm", "scikit-learn"])
# torch / torchvision / numpy / scipy / matplotlib / seaborn already ship with Colab


In [ ]:
# 3. Clone or update TSRNet
import os
import subprocess
if not os.path.exists("TSR_ECG"):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "-q", "https://github.com/SKYGOD07/TSR_ECG.git"])
else:
    print("Repository already exists. Pulling latest fixes...")
    subprocess.run(["git", "-C", "TSR_ECG", "pull"])

os.chdir("TSR_ECG")

## Step 1 — Preprocess raw PTB-XL into `train.npy` / `test.npy` / `label.npy`

This follows the same preprocessing TSRNet's authors used (from `MediaBrain-SJTU/ECGAD`):
- Loads `ptbxl_database.csv` + `scp_statements.csv` to get each record's diagnostic superclass
- Uses PTB-XL's official fold 10 as the held-out test set, all other folds as train
- **Train set = only `NORM` (healthy) records** — this is an anomaly-detection model, it never sees an abnormal ECG during training
- **Test set = all fold-10 records**, labeled `0` = normal, `1` = abnormal
- Reads the actual waveform signals via `wfdb` at 500 Hz (your `records500` folder)
- Applies bandpass/notch filtering (`heartpy`) and normalizes each lead to [-1, 1]

⚠️ This reads and filters ~21,000 signals — expect this cell to take a while (tens of minutes) depending on Drive read speed. It only needs to be run once; after that `train.npy`/`test.npy`/`label.npy` are saved to `data/` and you can skip straight to training in future sessions.


In [ ]:
import ast
import copy
import numpy as np
import pandas as pd
import wfdb
import heartpy as hp
from tqdm.auto import tqdm

os.makedirs('data', exist_ok=True)
SAMPLING_RATE = 500  # matches your records500 folder (TSRNet dataloader assumes 500 Hz)

def load_raw_data(df, sampling_rate, path):
    col = 'filename_lr' if sampling_rate == 100 else 'filename_hr'
    data = [wfdb.rdsamp(os.path.join(path, f)) for f in tqdm(df[col], desc='Reading WFDB signals')]
    return np.array([signal for signal, meta in data])

def preprocess_ptbxl(path, sampling_rate=500):
    Y = pd.read_csv(os.path.join(path, 'ptbxl_database.csv'), index_col='ecg_id')
    Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

    X = load_raw_data(Y, sampling_rate, path)

    agg_df = pd.read_csv(os.path.join(path, 'scp_statements.csv'), index_col=0)
    agg_df = agg_df[agg_df.diagnostic == True]

    def aggregate_diagnostic(y_dic):
        tmp = [agg_df.loc[k].diagnostic_class for k in y_dic.keys() if k in agg_df.index]
        return list(set(tmp))

    Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

    test_fold = 10
    X_train = X[np.where(Y.strat_fold != test_fold)]
    y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
    X_test = X[np.where(Y.strat_fold == test_fold)]
    y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

    train_data, count = [], 0
    for item in y_train:
        try:
            if item[0] == 'NORM':
                train_data.append(X_train[count])
            count += 1
        except Exception:
            count += 1
    train_data = np.asarray(train_data)

    test_label, test_data, count = [], [], 0
    for item in y_test:
        try:
            test_label.append(0 if item[0] == 'NORM' else 1)
            test_data.append(X_test[count])
            count += 1
        except Exception:
            count += 1
    test_label = np.asarray(test_label)
    test_data = np.asarray(test_data)

    print(f"train_data: {train_data.shape}, test_data: {test_data.shape}, "
          f"test_label: {test_label.shape} (positives/abnormal: {test_label.sum()})")
    return train_data, test_data, test_label

def normalize(X_ori):
    X = copy.deepcopy(X_ori)
    for n in range(X.shape[0]):
        for lead in range(12):
            seq = X[n][:, lead]
            X[n][:, lead] = 2 * (seq - seq.min()) / (seq.max() - seq.min()) - 1
    return X

def hp_preprocess(X):
    out = []
    for i in tqdm(range(X.shape[0]), desc='Filtering signals'):
        leads = []
        for lead in range(12):
            ecg = X[i][:, lead]
            f1 = hp.filter_signal(ecg, sample_rate=500, filtertype='highpass', cutoff=1)
            f2 = hp.filter_signal(f1, sample_rate=500, cutoff=35, filtertype='notch')
            f3 = hp.filter_signal(f2, sample_rate=500, filtertype='lowpass', cutoff=25)
            leads.append(f3)
        out.append(np.array(leads).T)
    return np.array(out)

def denoise_train(train_data):
    denoised = normalize(hp_preprocess(train_data))
    kept = []
    for i in range(denoised.shape[0]):
        try:
            hp.process(denoised[i, :, 1], 500.0)
        except Exception:
            continue
        kept.append(denoised[i])
    kept = np.array(kept)
    np.save('data/train.npy', kept)
    print("Saved data/train.npy", kept.shape)

def denoise_test(test_data, test_label):
    denoised = hp_preprocess(test_data)
    data_kept, label_kept = [], []
    for i in range(denoised.shape[0]):
        try:
            hp.process(denoised[i, :, 1], 500.0)
        except Exception:
            continue
        data_kept.append(denoised[i])
        label_kept.append(test_label[i])
    data_kept = np.array(data_kept)
    label_kept = np.array(label_kept)
    np.save('data/test.npy', data_kept)
    np.save('data/label.npy', label_kept)
    print("Saved data/test.npy", data_kept.shape, "and data/label.npy", label_kept.shape)

# Run preprocessing (only needs to happen once)
train_data, test_data, test_label = preprocess_ptbxl(PTBXL_DATA_PATH, SAMPLING_RATE)
print("Denoising + normalizing train set...")
denoise_train(train_data)
print("Denoising test set...")
denoise_test(test_data, test_label)


In [ ]:
# Sanity check on the produced files
tr = np.load('data/train.npy')
te = np.load('data/test.npy')
lb = np.load('data/label.npy')
print("train.npy:", tr.shape)   # expect (N, 5000, 12)
print("test.npy: ", te.shape)
print("label.npy:", lb.shape, "| abnormal count:", int(lb.sum()), "/", len(lb))


## (Optional) Back up the processed `.npy` files to Drive

Preprocessing is the slow part — copy the result to Drive so you never have to redo it.


In [ ]:
# In Kaggle, any files saved to /kaggle/working/ are kept as the notebook output.
# The data/ directory we created is already inside /kaggle/working/ (the default directory).
# So train.npy, test.npy, and label.npy will be saved automatically when you Save & Run All!


## Step 2 — Train TSRNet

Trains the multimodal (time + spectrogram) model on the normal-only training set. Checkpoints are saved to `ckpt/` every time validation AUC improves.

`--dims 12` = number of ECG leads (not signal length — the repo's own flag naming is a bit misleading). `--spec True` enables the spectrogram branch (the full model from the paper); drop it to train the lighter time-only model.

Lower `--epochs` for a quicker first run; the paper's default is 50.


In [ ]:
import os
os.system('python train.py --data_path data/ --dims 12 --spec True --epochs 30 --batch_size 32 --save_path ckpt/ --save_model 1')


In [ ]:
# See which checkpoint(s) got saved (filenames encode the epoch where AUC improved)
os.system("ls -la ckpt/")


## Step 3 — Get the ECG anomaly scores (AUC)

Pick the checkpoint with the highest epoch number (last one saved = best AUC during training, since `train.py` only saves when AUC improves) and run `test.py` with the Peak-based Error option for the paper's best-reported results.


In [ ]:
import glob
import os

def _epoch_key(p):
    try:
        return int(p.replace('\\', '/').split('/')[-1].split('-')[-1].split('.')[0])
    except ValueError:
        return -1

def find_best_ckpt():
    ckpts = sorted(glob.glob('ckpt/TSRNet-*.pt'), key=_epoch_key)
    numbered = [c for c in ckpts if _epoch_key(c) >= 0]
    if numbered:
        return numbered[-1]
    if os.path.exists('ckpt/TSRNet-latest.pt'):
        return 'ckpt/TSRNet-latest.pt'
    return None

# Auto-train if no checkpoint exists yet
if find_best_ckpt() is None:
    print('No checkpoint found — running training first (this may take a while)...')
    os.makedirs('ckpt', exist_ok=True)
    ret = os.system('python train.py --data_path data/ --dims 12 --spec True --epochs 30 --batch_size 32 --save_path ckpt/ --save_model 1')
    if ret != 0:
        raise RuntimeError('Training failed (non-zero exit code). Check the output above for errors.')

best_ckpt = find_best_ckpt()
if best_ckpt is None:
    raise FileNotFoundError(
        'Training completed but no checkpoint was saved. '
        'This can happen if AUC never improved and TSRNet-latest.pt was not written. '
        'Check that data/train.npy, data/test.npy, and data/label.npy exist.'
    )

print(f'Using checkpoint: {best_ckpt}')
cmd = f'python test.py --data_path data/ --dims 12 --spec True --mask_loss True --load_model 1 --load_path "{best_ckpt}"'
os.system(cmd)


The line printed above — `Detection AUC: 0.xxx` — is your ECG anomaly-detection score. Higher is better (1.0 = perfect separation of normal vs abnormal ECGs, 0.5 = random guessing). The paper reports ~0.86-0.87 AUC on PTB-XL with the full multimodal + Peak-based Error setup.

**Notes / things you can tweak:**
- Try `--mask_loss True` vs without it in the test step — the paper says Peak-based Error usually helps.
- Try without `--spec True` in training/testing to compare the lighter time-only model against the full multimodal one.
- If Drive I/O is the bottleneck during preprocessing, copying `records500/` to the Colab local disk first (`!cp -r ...`) before running Step 1 will speed it up a lot.
